In [2]:
!pip install mamba-ssm[causal-conv1d]
!pip install triton

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 2.2 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.5.0.post8-cp310-cp310-linux_x86_64.whl size=103960374 sha256=ec956a2d0fa48bd16a8dd5be9ad1c03db88565ea6ee2679e362940fef659284b
  Stored in directory: /root/.cache/pip/wheels/75/ef/0a/d9abf869acdd5fc07f403f4d8dd9db650cd66e81528a907941
  Created wheel for mamba-ssm: filename=mamba_ssm-2.2.4-cp310-cp310-linux_x86_64.whl size=323655716 sha256=0f4b4e95ce6271534b916f819bd33c1283d96208fbaac3f62a3c9777a3501187
  Stored in directory: /root/.cache/pip/wheels/aa/af/c7/fb77bfcd94bd3e052545033449d8c47dc97222d79c39c5bc67
Successfully built causal-conv1d mamba-ssm
  Using cached triton-3.2.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.4 kB)
Using cached tr

In [3]:
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm.modules.mamba_simple import Mamba
from torch.optim import Adam
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import GPT2Tokenizer
from mamba_ssm.models.mixer_seq_simple import MixerModel
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel
from transformers import AutoModelForCausalLM
from mamba_ssm.models.config_mamba import MambaConfig
from mamba_ssm.utils.generation import InferenceParams
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import nltk
from torch.amp import autocast, GradScaler
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
tokenizer_en = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer_frn = AutoTokenizer.from_pretrained("camembert-base")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
special_tokens_dict = {
    "sep_token": "<sep>",
    "pad_token": "<pad>",
}
tokenizer.add_special_tokens(special_tokens_dict)
tokenizer.save_pretrained("gpt2-with-special-tokens")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

('gpt2-with-special-tokens/tokenizer_config.json',
 'gpt2-with-special-tokens/special_tokens_map.json',
 'gpt2-with-special-tokens/vocab.json',
 'gpt2-with-special-tokens/merges.txt',
 'gpt2-with-special-tokens/added_tokens.json')

In [5]:


class Add_Norm(nn.Module):
    def __init__(self, d_model, dropout, residual, drop_flag=1):
        super(Add_Norm, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
        self.residual = residual
        self.drop_flag = drop_flag
    
    def forward(self, new, old):
        new = self.dropout(new) if self.drop_flag else new
        return self.norm(old + new) if self.residual else self.norm(new)

class BimambaEncoderLayer(nn.Module):
    def __init__(self, 
                 d_model,
                 d_conv,
                 d_state,
                 expand,
                 dropout=0.2,
                 d_ff=256, 
                 activation="relu", 
                 residual=1):
        super(BimambaEncoderLayer, self).__init__()
        self.d_model=d_model
        self.d_ff=d_ff
        self.d_conv=d_conv
        self.d_state=d_state
        self.expand=expand

        self.mamba_forward=Mamba(
            d_model=self.d_model,
            d_state=self.d_state,
            d_conv=self.d_conv,
            expand=self.expand,
        )
        self.addnorm_for=Add_Norm(d_model,dropout,residual=0,drop_flag=0)
        self.mamba_backward=Mamba(
            d_model=self.d_model,
            d_state=self.d_state,
            d_conv=self.d_conv,
            expand=self.expand,
        ) 
        self.addnorm_back=Add_Norm(d_model,dropout,residual=0,drop_flag=0)
        self.addnorm_output=Add_Norm(d_model,dropout,residual=1,drop_flag=0)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.addnorm_ffn = Add_Norm(d_model, dropout, residual, drop_flag=1)

    def forward(self, x):
        # [B, S, D]
        output_forward = self.mamba_forward(x)
        output_forward = self.addnorm_for(output_forward, x)
        output_backward = self.mamba_backward(x.flip(dims=[1])).flip(dims=[1])
        output_backward = self.addnorm_back(output_backward, x)
        output = output_forward + output_backward
        output = self.addnorm_output(output,x)
        temp = output
        output = self.feed_forward(output)
        output = self.addnorm_ffn(output, temp)
        return output


def createLayer(
    d_model,
    d_conv,
    d_state,
    expand,
    dropout
):
    layer = BimambaEncoderLayer(
        d_model,
        d_conv,
        d_state,
        expand,
        dropout
    )
    return layer

class Encoder(nn.Module):
    def __init__(
        self, 
        d_model,
        d_conv,
        d_state,
        expand,
        dropout,
        n_layer,
        vocab_size,
    ):
        super(Encoder, self).__init__()
        self.d_model=d_model
        self.d_conv=d_conv
        self.d_state=d_state
        self.expand=expand
        self.droput=dropout
        self.n_layer=n_layer
        self.vocab_size=vocab_size
        self.embedding = nn.Embedding(vocab_size,d_model)
        self.layers = nn.ModuleList(
            [
                createLayer(
                    d_model,
                    d_conv,
                    d_state,
                    expand,
                    dropout
                )
                for _ in range(n_layer)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)
    
    def forward(self,x):
        x=self.embedding(x)
        for layers in self.layers:
            x=layers(x)

        x=self.final_norm(x)

        return x
        

In [6]:
class DecoderLayer(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super(DecoderLayer,self).__init__()
        self.self_attention = nn.MultiheadAttention(d_model,n_heads,dropout=dropout)
        self.encoder_attention = nn.MultiheadAttention(d_model,n_heads,dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.ReLU(),
            nn.Linear(4*d_model, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, target, encoder_output, trg_mask=None, src_mask=None):
        target_transpose=target.transpose(0,1)
        # Self attention
        _target, _ = self.self_attention(target_transpose,
                                         target_transpose,
                                         target_transpose,
                                         attn_mask=trg_mask)
        _target=_target.transpose(0,1)
        target = self.norm1(target + self.dropout1(_target))
        # Encoder attention (cross attention)
        target_transpose=target.transpose(0,1)
        encoder_output_transpose=encoder_output.transpose(0,1)
        target, attn_weights = self.encoder_attention(
            target_transpose,
            encoder_output_transpose,
            encoder_output_transpose,
            attn_mask=src_mask
        )
        target=target.transpose(0,1)
        target = self.norm2(target + self.dropout2(_target))
        
        # Feed forward
        _target = self.ffn(target)
        target = self.norm3(target + self.dropout3(_target))
        
        return target

class Decoder(nn.Module):
    def __init__(self,vocab_size,d_model,n_heads,n_layer,dropout):
        super(Decoder,self).__init__()
        self.d_model=d_model
        self.vocab_size=vocab_size
        self.n_heads=n_heads
        self.n_layer=n_layer
        self.dropout=dropout

        self.embedding=nn.Embedding(vocab_size,d_model)
        self.position_encodding=nn.Parameter(torch.randn(1, 1, d_model))
        self.dropout=nn.Dropout(dropout)

        self.decoder_layers=nn.ModuleList([
            DecoderLayer(
                d_model,
                n_heads,
                dropout
            )
            for _ in range(n_layer)
        ])

        self.logits=nn.Linear(d_model,vocab_size)

    def forward(self,target,encoder_output,trg_mask=None,src_mask=None):
        
        Batch,length=target.shape
        target=self.embedding(target)*torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32))
        target+=self.position_encodding[:,:length,:]
        target=self.dropout(target)

        for layers in self.decoder_layers:
            target=layers(target,encoder_output,trg_mask,src_mask)

        output=self.logits(target)
        return output
        

In [7]:
d_model = 512
d_state = 64
d_conv=4
expand=2
n_heads=8
dropout=0.2

vocab_size=50000
encoder_n_layer=6
decoder_n_layer=6
model = Encoder(
    d_model=d_model,
    d_conv=d_conv,
    d_state=d_state,
    expand=expand,
    dropout=dropout,
    n_layer=encoder_n_layer,
    vocab_size=vocab_size
    
).to('cuda')

batch, length = 2, 64
x = torch.randint(0,49999,(batch, length)).to("cuda")
encoder_output=model(x)
print(encoder_output.shape)

decoder = Decoder(
    d_model=d_model,
    n_heads=n_heads,
    vocab_size=vocab_size,
    n_layer=decoder_n_layer,
    dropout=dropout
).to('cuda')
length=100
target = torch.randint(0,49999,(batch, length)).to("cuda")

decoder_output = decoder(target,encoder_output)

print(decoder_output.shape)

torch.Size([2, 64, 512])
torch.Size([2, 100, 50000])


In [8]:
dataset = load_dataset("opus_books", "en-fr")  # Returns a DatasetDict
train_data = dataset["train"]  # Extract the train split

max_en_length = max(len(ex["translation"]["en"].split()) for ex in train_data)
max_fr_length = max(len(ex["translation"]["fr"].split()) for ex in train_data)

# Shuffle the dataset
train_data = train_data.shuffle(seed=42)

# Define split sizes
train_size = int(0.8 * len(train_data))  # 80% training
val_size = int(0.1 * len(train_data))    # 10% validation
test_size = len(train_data) - train_size - val_size  # 10% test

# Split using .select()
train_dataset = train_data.select(range(train_size))
val_dataset = train_data.select(range(train_size, train_size + val_size))
test_dataset = train_data.select(range(train_size + val_size, len(train_data)))

print(f"Train: {len(train_dataset)}, Validation: {len(val_dataset)}, Test: {len(test_dataset)}")
def collate_fn(batch):
    en_texts = [ex["translation"]["en"] for ex in batch]  # Extract English texts
    fr_texts = [ex["translation"]["fr"] for ex in batch]  # Extract French texts

    # Tokenize source (English)
    en_inputs = tokenizer(
        en_texts, 
        padding="max_length",  
        truncation=True, 
        max_length=max_en_length, 
        return_tensors="pt"
    )

    # Tokenize target (French)
    fr_targets = tokenizer(
        fr_texts, 
        padding="max_length",  
        truncation=True, 
        max_length=max_fr_length,  
        return_tensors="pt"
    )

    return {
        "input_ids": en_inputs["input_ids"],
        "attention_mask": en_inputs["attention_mask"],
        "labels": fr_targets["input_ids"]
    }
    
epochs = 1
learning_rate = 1e-4
batch_size = 32

# Create DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
validation_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)  
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)  

print(len(train_dataloader))

README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

Train: 101668, Validation: 12708, Test: 12709
3178


In [9]:
# Hyperparameters
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
EPOCHS = 1
LEARNING_RATE = 5e-5
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
print(f"n_heads={n_heads}, decoder_n_layer={decoder_n_layer}")


print(f"d_model={d_model}, d_conv={d_conv}, d_state={d_state}, expand={expand}, dropout={dropout}, encoder_n_layer={encoder_n_layer}, vocab_size={vocab_size}")


# Initialize model
encoder = Encoder(d_model, d_conv, d_state, expand, dropout, encoder_n_layer, vocab_size).to(DEVICE)
decoder = Decoder(vocab_size, d_model, n_heads, decoder_n_layer, dropout).to(DEVICE)

# Define loss function (ignore padding index)
PAD_ID = 0  # Update based on your tokenizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

# Define optimizer
optimizer = optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=LEARNING_RATE)

# Scheduler (optional)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.95)
#loop

# Debug: Print before training to check the range of token IDs
# Debug: Print before training to check the range of token IDs
GRADIENT_ACCUMULATION_STEPS = 4  # Use gradient accumulation
# Mixed precision training
scaler = GradScaler()

for epoch in range(EPOCHS):
    # Training phase
    encoder.train()
    decoder.train()
    
    total_train_loss = 0
    optimizer.zero_grad()  # Reset gradients at the start of each epoch

    # Wrap the training dataloader with tqdm for a progress bar
    train_progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{EPOCHS} [Training]", leave=False)

    for step, batch in enumerate(train_progress_bar):
        # Unpack the batch
        src_input_ids = batch['input_ids'].to(DEVICE) 
        tgt_input_ids = batch['labels'].to(DEVICE)  
        attention_mask = batch['attention_mask'].to(DEVICE)

        # Clamp input IDs to valid vocabulary range
        src_input_ids = torch.clamp(src_input_ids, max=vocab_size - 1).to(DEVICE)
        tgt_input_ids = torch.clamp(tgt_input_ids, max=vocab_size - 1).to(DEVICE)

        # Create decoder input by shifting tgt_input_ids
        tgt_input = tgt_input_ids[:, :-1]  # Shift left for decoder input
        tgt_output = tgt_input_ids[:, 1:]  # Shift right for target labels

      
        with autocast(device_type='cuda'): 
            encoder_output = encoder(src_input_ids)
            encoder_output = encoder_output[:, :tgt_input.size(1), :] 
            logits = decoder(tgt_input, encoder_output)

            # Compute loss
            loss = criterion(
                logits.reshape(-1, logits.size(-1)), 
                tgt_output.reshape(-1)  
            )
            loss = loss / GRADIENT_ACCUMULATION_STEPS  

        # Backpropagation
        scaler.scale(loss).backward()  #

        # Gradient accumulation
        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(decoder.parameters()), max_norm=1.0)

            # Optimizer step and scaler update
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()  # Reset gradients

        total_train_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS  # Scale loss back up

        # Update progress bar description with current loss
        train_progress_bar.set_postfix({"Train Loss": loss.item()})

        # Clear unused GPU memory
        if step % 10 == 0:
            torch.cuda.empty_cache()

    # Compute average training loss for the epoch
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1}/{EPOCHS}, Train Loss: {avg_train_loss:.4f}")

    # Validation phase
    encoder.eval()
    decoder.eval()
    
    total_val_loss = 0

    # Wrap the validation dataloader with tqdm for a progress bar
    val_progress_bar = tqdm(validation_dataloader, desc=f"Epoch {epoch + 1}/{EPOCHS} [Validation]", leave=False)

    with torch.no_grad():  
        for batch in val_progress_bar:
            # Unpack the batch
            src_input_ids = batch['input_ids'].to(DEVICE)  
            tgt_input_ids = batch['labels'].to(DEVICE) 
            attention_mask = batch['attention_mask'].to(DEVICE)

            # Clamp input IDs to valid vocabulary range
            src_input_ids = torch.clamp(src_input_ids, max=vocab_size - 1).to(DEVICE)
            tgt_input_ids = torch.clamp(tgt_input_ids, max=vocab_size - 1).to(DEVICE)

            # Create decoder input by shifting tgt_input_ids
            tgt_input = tgt_input_ids[:, :-1]  # Shift left for decoder input
            tgt_output = tgt_input_ids[:, 1:]  # Shift right for target labels

            # Forward pass through encoder and decoder
            with autocast(device_type='cuda'):  # Mixed precision
                encoder_output = encoder(src_input_ids)
                encoder_output = encoder_output[:, :tgt_input.size(1), :]  # Truncate to match target sequence length
                logits = decoder(tgt_input, encoder_output)

                # Compute loss
                loss = criterion(
                    logits.reshape(-1, logits.size(-1)),  # Reshape logits
                    tgt_output.reshape(-1)  # Reshape target labels
                )

            total_val_loss += loss.item()

            # Update progress bar description with current loss
            val_progress_bar.set_postfix({"Val Loss": loss.item()})

    # Compute average validation loss for the epoch
    avg_val_loss = total_val_loss / len(validation_dataloader)
    print(f"Epoch {epoch + 1}/{EPOCHS}, Val Loss: {avg_val_loss:.4f}")

    # Step the scheduler (optional)
    scheduler.step()




Tokenizer vocab size: 50257
Using device: cuda
CUDA available: True
n_heads=8, decoder_n_layer=6
d_model=512, d_conv=4, d_state=64, expand=2, dropout=0.2, encoder_n_layer=6, vocab_size=50000


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.93 GiB. GPU 0 has a total capacity of 14.74 GiB of which 1.92 GiB is free. Process 2477 has 12.82 GiB memory in use. Of the allocated memory 12.47 GiB is allocated by PyTorch, and 219.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import torch
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
from tqdm import tqdm
import nltk

# Download NLTK data for tokenization (if not already downloaded)
nltk.download('punkt')

# Set model to evaluation mode
encoder.eval()
decoder.eval()

# Function to generate predictions
def generate_predictions(encoder, decoder, dataloader, tokenizer, device):
    predictions = []
    references = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Generating Predictions"):
            # Unpack the batch
            src_input_ids = batch['input_ids'].to(DEVICE)  
            tgt_input_ids = batch['labels'].to(DEVICE) 
            attention_mask = batch['attention_mask'].to(DEVICE)

             # Create decoder input by shifting tgt_input_ids
            tgt_input = tgt_input_ids[:, :-1]  # Shift left for decoder input
            tgt_output = tgt_input_ids[:, 1:]  # Shift right for target labels

            # Generate predictions
            encoder_output = encoder(src_input_ids)
            encoder_output = encoder_output[:, :tgt_input.size(1), :] 
            logits = decoder(tgt_input, encoder_output)
            preds = torch.argmax(logits, dim=-1)  # Get predicted token IDs

            # Convert predictions and references to text
            for pred, ref in zip(preds, tgt_input_ids):
                predictions.append(tokenizer.decode(pred.tolist(), skip_special_tokens=True))
                references.append([tokenizer.decode(ref.tolist(), skip_special_tokens=True)])  # Wrap in list for corpus_bleu

    return predictions, references

# Generate predictions and references
predictions, references = generate_predictions(encoder, decoder, test_dataloader, tokenizer, DEVICE)

# Calculate BLEU score
def calculate_bleu_score(predictions, references):
    # Tokenize predictions and references
    predictions_tokens = [nltk.word_tokenize(pred) for pred in predictions]
    references_tokens = [[nltk.word_tokenize(ref[0])] for ref in references]  # Wrap in list for corpus_bleu

    # Calculate corpus BLEU score
    bleu_score = corpus_bleu(references_tokens, predictions_tokens)
    return bleu_score

# Compute BLEU score
bleu_score = calculate_bleu_score(predictions, references)
print(f"BLEU Score: {bleu_score:.4f}")